In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../data/processed/rumah_clean.csv")
TARGET = "price_in_rp"
print(df.shape)

(3424, 24)


In [3]:
BUANG = ["url", "ads_id", "title", "address", "property_type", "facilities"]
df = df.drop(columns=BUANG)

In [4]:
df["listrik"] = pd.to_numeric(df["electricity"].str.extract(r"(\d+)")[0], errors="coerce")
df = df.drop(columns="electricity")

In [5]:
df["certificate"] = df["certificate"].replace(
    {"hp - hak pakai": "lainnya (ppjb,girik,adat,dll)"}
)
print(df["certificate"].value_counts(dropna=False))

certificate
shm - sertifikat hak milik       2892
hgb - hak guna bangunan           199
lainnya (ppjb,girik,adat,dll)     196
NaN                               137
Name: count, dtype: int64


In [6]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train = X_train.copy()
X_test = X_test.copy()
print(X_train.shape, X_test.shape)

(2739, 17) (685, 17)


In [7]:
(X_train.isnull().mean() * 100).round(2).sort_values(ascending=False).head(10)

building_orientation    46.59
building_age            41.04
furnishing              10.95
listrik                  7.96
property_condition       6.79
certificate              4.16
bedrooms                 0.84
bathrooms                0.73
floors                   0.07
district                 0.00
dtype: float64

In [8]:
KAT_KOSONG = ["certificate", "furnishing", "property_condition", "building_orientation"]
for col in KAT_KOSONG:
    X_train[col] = X_train[col].fillna("tidak diketahui")
    X_test[col] = X_test[col].fillna("tidak diketahui")

In [9]:
NUM_PENANDA = ["building_age", "listrik"]
NUM_SEDIKIT = ["bedrooms", "bathrooms", "floors"]

# 1) hitung median dari TRAIN saja
nilai_isi = {col: X_train[col].median() for col in NUM_PENANDA + NUM_SEDIKIT}

# 2) buat penanda "dulunya kosong" SEBELUM diisi
for col in NUM_PENANDA:
    X_train[col + "_kosong"] = X_train[col].isnull().astype(int)
    X_test[col + "_kosong"] = X_test[col].isnull().astype(int)

# 3) isi, pakai angka yang sama untuk train dan test
for col, v in nilai_isi.items():
    X_train[col] = X_train[col].fillna(v)
    X_test[col] = X_test[col].fillna(v)

print(nilai_isi)

{'building_age': np.float64(0.0), 'listrik': np.float64(2200.0), 'bedrooms': np.float64(3.0), 'bathrooms': np.float64(2.0), 'floors': np.float64(2.0)}


In [10]:
print(X_train.isnull().sum().sum(), X_test.isnull().sum().sum())

0 0


In [11]:
hitung = X_train["district"].value_counts()
district_ramai = hitung[hitung >= 10].index      # dihitung dari TRAIN

for d in (X_train, X_test):
    d["district"] = d["district"].where(d["district"].isin(district_ramai), "lainnya")

print(len(district_ramai), "district dipertahankan")
print("porsi 'lainnya' di train:", (X_train["district"] == "lainnya").mean().round(3))

72 district dipertahankan
porsi 'lainnya' di train: 0.285


In [12]:
KAT = ["city", "district", "certificate", "furnishing",
       "property_condition", "building_orientation"]

X_train_enc = pd.get_dummies(X_train, columns=KAT).astype(float)
X_test_enc = pd.get_dummies(X_test, columns=KAT).astype(float)

# samakan kolom test dengan train
X_test_enc = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)
print(X_train_enc.shape, X_test_enc.shape)

(2739, 118) (685, 118)


In [13]:
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)
# kembali ke rupiah nanti: np.expm1(prediksi)

In [14]:
import os, joblib
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

X_train_enc.to_csv("../data/processed/X_train.csv", index=False)
X_test_enc.to_csv("../data/processed/X_test.csv", index=False)
y_train_log.to_csv("../data/processed/y_train_log.csv", index=False)
y_test_log.to_csv("../data/processed/y_test_log.csv", index=False)

# catatan nilai isi, dipakai lagi nanti oleh aplikasi Streamlit
joblib.dump({
    "nilai_isi": nilai_isi,
    "district_ramai": list(district_ramai),
    "kolom": list(X_train_enc.columns),
}, "../models/preprocess_info.joblib")

['../models/preprocess_info.joblib']